# 11b — Busca de hiperparâmetros (CNN e fusão) para melhorar a acurácia

Rode **depois do notebook 11** (usa o LSTM já ajustado, que teve F1 = 1,0 na validação e não é refeito aqui).

**Objetivo**: reduzir a confusão Normal → Atenção (23% das janelas normais no teste), que derrubou a acurácia (0,832 < 0,90 de H2).

**O que é variado** (a arquitetura não muda, então o firmware v5 continua servindo):

| Etapa | Hiperparâmetro | Valores |
|---|---|---|
| CNN | taxa de aprendizado | 1e-4 (v1), 3e-4 |
| CNN | peso das classes | inverso da frequência (v1), raiz do inverso |
| CNN | janelas normais por sessão | 4000 (v1), 8000 |
| Fusão | peso das classes | inverso (v1), raiz do inverso, nenhum |
| Fusão | deslocamento δ do viés da classe Normal | 0 (v1) a 3, passo 0,25 |

São 8 CNNs × 3 fusões × 13 valores de δ = 312 combinações. O δ soma-se ao viés `b[0]` da camada de fusão: no firmware é só o número em `FUSAO_B[0]`.

**Regras de rigor** (declarar no texto):
1. A seleção usa **só a sessão de validação** (SIM005). A sessão de teste (SIM006) **não é lida** aqui.
2. Critério de seleção, fixado antes de rodar: entre as combinações que, na validação, têm F1 macro ≥ 0,80 e recall de Falha ≥ 0,85, escolhe-se a de **maior acurácia** (empate: maior F1 macro). São os limiares de H2. Se nenhuma atender, escolhe-se a de maior F1 macro, e isso deve ser relatado.
3. A primeira avaliação no teste (v1) já foi vista; relatar os dois resultados (v1 e v2), não só o melhor.
4. Os modelos v1 ficam guardados em `modelos/sim_v1/`. Os escolhidos substituem os de `modelos/sim/`, e o notebook 12 e o 05b rodam sem mudança.

Tempo estimado no Colab com GPU: 8 treinos da CNN (cerca de 3 a 6 min cada) + 24 fusões (rápidas).

In [1]:
# 1. Ambiente e caminhos (igual ao notebook 11)
import os, sys, json, subprocess, importlib.util
if importlib.util.find_spec("tensorflow") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"], check=True)
import numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/dissertacao_tinyml"
except ImportError:
    BASE = os.environ.get("BASE_TINYML", "dissertacao_tinyml")
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter
CONJUNTO = os.environ.get("CONJUNTO_TINYML", "coleta_sim")   # "coleta_sim" = sessões sintéticas (notebook 07b) | "coleta" = dados do nó
SIM = CONJUNTO != "coleta"
print("Conjunto de dados:", CONJUNTO)
COLETA, ROT = f"{BASE}/{CONJUNTO}", f"{BASE}/{CONJUNTO}/rotulado"
CNN0, LSTM0, SAIDA = f"{BASE}/modelos/cnn_v3", f"{BASE}/modelos/lstm", f"{BASE}/modelos/" + ("sim" if SIM else "insitu")
os.makedirs(SAIDA, exist_ok=True)
import shutil, itertools, time
BACKUP = SAIDA + "_v1"
if not os.path.exists(BACKUP):
    shutil.copytree(SAIDA, BACKUP); print("Modelos v1 guardados em", BACKUP)
else:
    print("Backup v1 já existe:", BACKUP)
os.makedirs(f"{SAIDA}/busca", exist_ok=True)

Mounted at /content/drive
Conjunto de dados: coleta_sim
Modelos v1 guardados em /content/drive/MyDrive/dissertacao_tinyml/modelos/sim_v1


In [2]:
# 2. Parâmetros e grade (fixados antes de rodar)
SEMENTE = 0
HP = {"lote": 64, "epocas": int(os.environ.get("EPOCAS_TINYML", 50)), "epocas_fusao": 300, "paciencia": 8, "lr_fusao": 1e-2}
GRADE_CNN = [dict(lr=lr, peso=p, max_normal=m) for m in (4000, 8000) for p in ("inversa", "raiz") for lr in (1e-4, 3e-4)]
PESO_FUSAO = ["inversa", "raiz", "nenhum"]
DELTAS = np.round(np.arange(0, 3.01, 0.25), 2)
CRIT = {"f1_min": 0.80, "recall_falha_min": 0.85}       # limiares de H2; maximiza-se a acurácia
MAX_NORMAL_SESSAO = 4000
tf.keras.utils.set_random_seed(SEMENTE); rng = np.random.default_rng(SEMENTE)
print(len(GRADE_CNN), "CNNs ×", len(PESO_FUSAO), "fusões ×", len(DELTAS), "deltas =", len(GRADE_CNN) * len(PESO_FUSAO) * len(DELTAS), "combinações")

8 CNNs × 3 fusões × 13 deltas = 312 combinações


In [3]:
# 3. Rótulos e matriz da Figura 5 (iguais aos do notebook 10)
# ===== Rotulação das sessões de eventos (seção 3.3.1 e Figura 5) =====
TIPOS = ["termico", "desbalanceamento", "obstrucao", "combinado"]
ACUSTICOS = {"desbalanceamento", "obstrucao", "combinado"}     # o tipo térmico não altera o nível acústico
TERMICOS = {"termico", "combinado"}
NOMES_CLASSE = ["Normal", "Atenção", "Crítico", "Falha"]

def classe_termica(t):
    t = np.asarray(t, dtype=float)
    return np.select([t > 35.0, t > 32.0, t > 27.0], [3, 2, 1], 0)

def classe_fusao_regra(nt, na):
    """Figura 5: prevalece o mais severo; térmico 2 com acústico >= 1 vira Falha; nível 3 em qualquer ramo = Falha."""
    nt, na = np.asarray(nt), np.asarray(na)
    c = np.maximum(nt, na)
    c = np.where((nt == 2) & (na >= 1), 3, c)
    return np.where((nt == 3) | (na == 3), 3, c).astype(int)

def eventos_da_sessao(evt, planilha, sessao):
    """Junta os pares início/fim do EVT.CSV com a planilha (tipo, nível, fundo) pelo número do evento."""
    e = evt[evt.evento > 0]
    ini = e[e.tipo == "inicio"].set_index("evento").amostra
    fim = e[e.tipo == "fim"].set_index("evento").amostra
    ev = pd.DataFrame({"amostra_ini": ini, "amostra_fim": fim}).reset_index()
    p = planilha[planilha.sessao == sessao].copy()
    problemas = []
    if ev.amostra_fim.isna().any():
        problemas.append(f"eventos sem fim no EVT.CSV: {ev[ev.amostra_fim.isna()].evento.tolist()}")
    if len(ev) != len(p):
        problemas.append(f"EVT.CSV tem {len(ev)} eventos e a planilha tem {len(p)} para {sessao}")
    ev = ev.merge(p[["evento", "tipo", "nivel", "fundo"]], on="evento", how="left")
    if ev.tipo.isna().any():
        problemas.append(f"eventos sem linha na planilha: {ev[ev.tipo.isna()].evento.tolist()}")
    return ev, problemas

def rotulos_acusticos(n_jan, ev, fs, transicao_s):
    """Nível acústico de cada janela de 1 s: nível do evento (tipos acústicos) ou 0; -1 = janela descartada
    por estar a menos de transicao_s de uma marcação. Também devolve o número do evento de cada janela (0 = fora)."""
    ini = np.arange(n_jan) * fs; fim = ini + fs
    y = np.zeros(n_jan, int); evid = np.zeros(n_jan, int)
    marg = int(transicao_s * fs)
    for _, e in ev.iterrows():
        a, b = int(e.amostra_ini), int(e.amostra_fim)
        dentro = (ini >= a) & (fim <= b)
        evid[dentro] = e.evento
        if e.tipo in ACUSTICOS: y[dentro] = int(e.nivel)
        for m in (a, b):                                   # janelas perto de uma marcação
            y[(fim > m - marg) & (ini < m + marg)] = -1
    return y, evid

def referencia_10s(termo, inicio_real, blocos, offset_s=0.0, max_lacuna_s=60):
    """Temperatura do termo-higrômetro interpolada na grade de 10 s do áudio (NaN onde não há referência)."""
    if termo is None or not len(termo):
        return np.full(len(blocos), np.nan)
    s = (pd.to_datetime(termo.horario) - pd.Timestamp(inicio_real)).dt.total_seconds().values + offset_s
    t = termo.temp_ref.values.astype(float)
    ordem = np.argsort(s); s, t = s[ordem], t[ordem]
    grade = np.asarray(blocos) * 10.0 + 5.0                # centro de cada bloco de 10 s do áudio
    ref = np.interp(grade, s, t, left=np.nan, right=np.nan)
    pos = np.searchsorted(s, grade)
    viz = np.minimum(np.abs(grade - s[np.clip(pos - 1, 0, len(s) - 1)]), np.abs(s[np.clip(pos, 0, len(s) - 1)] - grade))
    ref[viz > max_lacuna_s] = np.nan                        # longe de qualquer leitura: sem referência
    return ref


In [4]:
# 4. Leitura das sessões rotuladas (sem as de teste) e normalização pela linha de base acústica (base.csv)
SESSOES = pd.read_csv(f"{COLETA}/sessoes.csv", dtype=str).fillna("")
SESSOES = SESSOES[SESSOES.papel.isin(["treino", "validacao"])]
if SIM:   # proteção: com o conjunto sintético, só entram sessões geradas pelo 07b (SIMnnn)
    _fora = [s for s in SESSOES.sessao if not str(s).startswith("SIM")]
    assert not _fora, f"sessoes.csv de {CONJUNTO} contém sessões que não são do 07b: {_fora}"
print("Sessões lidas:", SESSOES.sessao.tolist())
b = pd.read_csv(f"{COLETA}/linha_de_base/base.csv")
MU, SD = b.media.values.astype(np.float32), b.desvio.values.astype(np.float32)
D = {}
for _, sx in SESSOES.iterrows():
    f = f"{ROT}/{sx.sessao}.npz"
    if not os.path.exists(f): print("Sem arquivo rotulado (rode o notebook 10):", sx.sessao); continue
    D[sx.sessao] = dict(np.load(f)); D[sx.sessao]["papel"] = sx.papel
print({s: d["papel"] for s, d in D.items()})
assert any(d["papel"] == "treino" for d in D.values()) and any(d["papel"] == "validacao" for d in D.values())

def z(X): return ((X.astype(np.float32) - MU) / SD)[..., None]

def conjunto_cnn(papel):
    X, y = [], []
    for s, d in D.items():
        if d["papel"] != papel: continue
        ya = d["ya"]; idx_a = np.where(ya > 0)[0]; idx_n = np.where(ya == 0)[0]
        if len(idx_n) > MAX_NORMAL_SESSAO: idx_n = rng.choice(idx_n, MAX_NORMAL_SESSAO, replace=False)
        idx = np.sort(np.concatenate([idx_a, idx_n]))
        X.append(z(d["Xc"][idx])); y.append(ya[idx])
    return np.concatenate(X), np.concatenate(y)
Xc_tr, yc_tr = conjunto_cnn("treino"); Xc_va, yc_va = conjunto_cnn("validacao")
print("CNN  treino:", np.bincount(yc_tr, minlength=4), " validação:", np.bincount(yc_va, minlength=4))

Sessões lidas: ['SIM001', 'SIM002', 'SIM003', 'SIM004', 'SIM005']
{'SIM001': 'treino', 'SIM002': 'treino', 'SIM003': 'treino', 'SIM004': 'treino', 'SIM005': 'validacao'}
CNN  treino: [16000  4860  4860  4860]  validação: [4000 1620 1620 1620]


In [5]:
# 5. Ramo térmico fixo (LSTM do notebook 11) e entradas da fusão que não dependem da CNN
FAIXAS = {"temp": (10.0, 45.0), "umid": (0.0, 100.0), "dpress": (-5.0, 5.0)}
def atributos(v):
    v = np.asarray(v, np.float32); dp = v[..., 2] - v[..., 2].mean(axis=-1, keepdims=True)
    cols = [v[..., 0], v[..., 1], dp]
    return np.stack([np.clip((c - lo) / (hi - lo), 0, 1) for c, (lo, hi) in zip(cols, FAIXAS.values())], -1).astype(np.float32)
def pesos(y, esquema="inversa"):
    if esquema == "nenhum": return None
    c = np.bincount(y, minlength=4); w = {k: len(y) / (np.count_nonzero(c) * n) for k, n in enumerate(c) if n > 0}
    return {k: v ** 0.5 for k, v in w.items()} if esquema == "raiz" else w
lstm = tf.keras.models.load_model(f"{BACKUP}/lstm_insitu.keras")
PRE = {}
for s, d in D.items():
    if not len(d["dec_y"]): continue
    jan = np.unique(np.concatenate([np.arange(j - 9, j + 1) for j in d["dec_j"]]))
    PRE[s] = dict(papel=d["papel"], jan=jan, Xz=z(d["Xc"][jan]), y=d["dec_y"].astype(int),
                  pl=lstm.predict(atributos(d["Xl"][d["dec_l"]]), batch_size=256, verbose=0))
print({s: len(p["y"]) for s, p in PRE.items()}, "decisões por sessão")

def entradas(modelo_cnn, papel):
    X, Y = [], []
    for s, p in PRE.items():
        if p["papel"] != papel: continue
        pc = np.zeros((len(D[s]["ya"]), 4), np.float32); pc[p["jan"]] = modelo_cnn.predict(p["Xz"], batch_size=256, verbose=0)
        pc10 = np.stack([pc[j - 9:j + 1].mean(0) for j in D[s]["dec_j"]])
        X.append(np.concatenate([p["pl"], pc10], 1)); Y.append(p["y"])
    return np.concatenate(X).astype(np.float32), np.concatenate(Y)

def metricas(y, yp):
    cm = confusion_matrix(y, yp, labels=[0, 1, 2, 3]); rec = cm.diagonal() / np.maximum(cm.sum(1), 1)
    return dict(acuracia=accuracy_score(y, yp), f1_macro=f1_score(y, yp, average="macro"),
                recall_Normal=rec[0], recall_Atencao=rec[1], recall_Critico=rec[2], recall_Falha=rec[3])

{'SIM001': 8611, 'SIM002': 1998, 'SIM003': 1998, 'SIM004': 1998, 'SIM005': 1998} decisões por sessão


In [6]:
# 6. Busca: CNN → fusão → deslocamento δ (tudo avaliado só na validação)
parada = lambda: tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=HP["paciencia"], restore_best_weights=True)
semente_cnn = json.load(open(f"{CNN0}/cnn_int8_quantizacao.json"))["semente"]
LINHAS, FUSOES = [], {}
for ic, g in enumerate(GRADE_CNN):
    t0 = time.time()
    MAX_NORMAL_SESSAO = g["max_normal"]; rng = np.random.default_rng(SEMENTE)
    Xc_tr, yc_tr = conjunto_cnn("treino"); Xc_va, yc_va = conjunto_cnn("validacao")
    tf.keras.utils.set_random_seed(SEMENTE)
    base_cnn = tf.keras.models.load_model(f"{CNN0}/cnn_semente{semente_cnn}.keras")
    cnn = tf.keras.Model(base_cnn.input, tf.keras.layers.Dense(4, activation="softmax", name="saida")(base_cnn.get_layer("densa").output), name="cnn2d_insitu")
    cnn.compile(optimizer=tf.keras.optimizers.Adam(g["lr"]), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h = cnn.fit(Xc_tr, yc_tr, validation_data=(Xc_va, yc_va), epochs=HP["epocas"], batch_size=HP["lote"],
                class_weight=pesos(yc_tr, g["peso"]), callbacks=[parada()], verbose=0)
    f1_cnn = f1_score(yc_va, cnn.predict(Xc_va, verbose=0).argmax(1), average="macro")
    cnn.save(f"{SAIDA}/busca/cnn_cfg{ic}.keras")
    Xf_tr, yf_tr = entradas(cnn, "treino"); Xf_va, yf_va = entradas(cnn, "validacao")
    for pf in PESO_FUSAO:
        tf.keras.utils.set_random_seed(SEMENTE)
        ent = tf.keras.Input((8,), name="prob_ramos")
        fus = tf.keras.Model(ent, tf.keras.layers.Dense(4, activation="softmax", name="fusao")(ent), name="fusao")
        fus.compile(optimizer=tf.keras.optimizers.Adam(HP["lr_fusao"]), loss="sparse_categorical_crossentropy")
        fus.fit(Xf_tr, yf_tr, validation_data=(Xf_va, yf_va), epochs=HP["epocas_fusao"], batch_size=HP["lote"], class_weight=pesos(yf_tr, pf),
                callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)], verbose=0)
        W, b = fus.get_layer("fusao").get_weights(); FUSOES[(ic, pf)] = (W, b)
        logit = Xf_va @ W + b
        for dl in DELTAS:
            lg = logit.copy(); lg[:, 0] += dl
            LINHAS.append(dict(cfg_cnn=ic, **g, epocas_cnn=len(h.history["loss"]), f1_cnn_val=f1_cnn, peso_fusao=pf, delta=dl,
                               **metricas(yf_va, lg.argmax(1))))
    print(f"CNN {ic} {g} | épocas {len(h.history['loss'])} | F1 CNN val {f1_cnn:.3f} | {time.time() - t0:.0f} s")
R = pd.DataFrame(LINHAS); R.to_csv(f"{SAIDA}/busca_hiperparametros.csv", index=False)
print("Combinações avaliadas:", len(R))

CNN 0 {'lr': 0.0001, 'peso': 'inversa', 'max_normal': 4000} | épocas 50 | F1 CNN val 0.772 | 276 s
CNN 1 {'lr': 0.0003, 'peso': 'inversa', 'max_normal': 4000} | épocas 50 | F1 CNN val 0.787 | 447 s
CNN 2 {'lr': 0.0001, 'peso': 'raiz', 'max_normal': 4000} | épocas 50 | F1 CNN val 0.770 | 267 s
CNN 3 {'lr': 0.0003, 'peso': 'raiz', 'max_normal': 4000} | épocas 50 | F1 CNN val 0.794 | 252 s
CNN 4 {'lr': 0.0001, 'peso': 'inversa', 'max_normal': 8000} | épocas 50 | F1 CNN val 0.738 | 327 s
CNN 5 {'lr': 0.0003, 'peso': 'inversa', 'max_normal': 8000} | épocas 33 | F1 CNN val 0.756 | 266 s
CNN 6 {'lr': 0.0001, 'peso': 'raiz', 'max_normal': 8000} | épocas 50 | F1 CNN val 0.762 | 327 s
CNN 7 {'lr': 0.0003, 'peso': 'raiz', 'max_normal': 8000} | épocas 50 | F1 CNN val 0.780 | 326 s
Combinações avaliadas: 312


In [7]:
# 7. Seleção pelo critério fixado (validação) e comparação com a configuração v1
v1 = R[(R.lr == 1e-4) & (R.peso == "inversa") & (R.max_normal == 4000) & (R.peso_fusao == "inversa") & (R.delta == 0)].iloc[0]
ok = R[(R.f1_macro >= CRIT["f1_min"]) & (R.recall_Falha >= CRIT["recall_falha_min"])]
if len(ok): esc = ok.sort_values(["acuracia", "f1_macro"], ascending=False).iloc[0]
else:      # reserva: sem combinação nos limiares, prioriza o equilíbrio entre classes (não a acurácia)
    esc = R.sort_values(["f1_macro", "recall_Falha"], ascending=False).iloc[0]
    print("ATENÇÃO: nenhuma combinação atende os limiares na validação; escolhida a de maior F1 macro. Relatar no texto.")
cols = ["cfg_cnn", "lr", "peso", "max_normal", "peso_fusao", "delta", "acuracia", "f1_macro", "recall_Normal", "recall_Atencao", "recall_Critico", "recall_Falha"]
print("Configuração v1 (validação):"); print(v1[cols].to_string())
print("\nEscolhida (validação):"); print(esc[cols].to_string())
print("\n10 melhores" + (" que atendem os limiares:" if len(ok) else " (nenhuma atende os limiares):"))
print((ok if len(ok) else R).sort_values(["acuracia", "f1_macro"], ascending=False)[cols].head(10).round(4).to_string(index=False))
# efeito isolado de δ na configuração escolhida (para o texto)
print("\nEfeito de δ (CNN e fusão escolhidas):")
print(R[(R.cfg_cnn == esc.cfg_cnn) & (R.peso_fusao == esc.peso_fusao)][["delta", "acuracia", "f1_macro", "recall_Normal", "recall_Falha"]].round(4).to_string(index=False))

Configuração v1 (validação):
cfg_cnn                  0
lr                  0.0001
peso               inversa
max_normal            4000
peso_fusao         inversa
delta                  0.0
acuracia           0.89039
f1_macro          0.891085
recall_Normal     0.873684
recall_Atencao    0.886686
recall_Critico    0.895522
recall_Falha      0.990431

Escolhida (validação):
cfg_cnn                  1
lr                  0.0003
peso               inversa
max_normal            4000
peso_fusao            raiz
delta                 0.75
acuracia          0.946446
f1_macro          0.935841
recall_Normal     0.994332
recall_Atencao     0.76204
recall_Critico    0.930348
recall_Falha      0.990431

10 melhores que atendem os limiares:
 cfg_cnn     lr    peso  max_normal peso_fusao  delta  acuracia  f1_macro  recall_Normal  recall_Atencao  recall_Critico  recall_Falha
       1 0.0003 inversa        4000       raiz   0.75    0.9464    0.9358         0.9943          0.7620          0.9303      

In [8]:
# 8. Grava os modelos escolhidos em modelos/sim (o notebook 12 e o 05b leem daqui)
ic, pf, dl = int(esc.cfg_cnn), esc.peso_fusao, float(esc.delta)
cnn = tf.keras.models.load_model(f"{SAIDA}/busca/cnn_cfg{ic}.keras"); cnn.save(f"{SAIDA}/cnn_insitu.keras")
W, b = FUSOES[(ic, pf)]; b = b.copy(); b[0] += dl
ent = tf.keras.Input((8,), name="prob_ramos")
fusao = tf.keras.Model(ent, tf.keras.layers.Dense(4, activation="softmax", name="fusao")(ent), name="fusao")
fusao.get_layer("fusao").set_weights([W, b]); fusao.save(f"{SAIDA}/fusao.keras")
json.dump({"W": W.tolist(), "b": b.tolist(), "entrada": "p_lstm[0..3], media10_p_cnn[0..3]", "delta_normal": dl, "peso_fusao": pf},
          open(f"{SAIDA}/fusao.json", "w"), indent=1)
shutil.copy(f"{BACKUP}/lstm_insitu.keras", f"{SAIDA}/lstm_insitu.keras")
# conferência: a fusão gravada reproduz a métrica da busca
Xf_va, yf_va = entradas(cnn, "validacao")
print("Validação com os modelos gravados:", {k: round(v, 4) for k, v in metricas(yf_va, fusao.predict(Xf_va, verbose=0).argmax(1)).items()})

Validação com os modelos gravados: {'acuracia': 0.9464, 'f1_macro': 0.9358, 'recall_Normal': np.float64(0.9943), 'recall_Atencao': np.float64(0.762), 'recall_Critico': np.float64(0.9303), 'recall_Falha': np.float64(0.9904)}


In [9]:
# 9. Quantização int8 da CNN escolhida (igual ao notebook 11) e concordância
g = GRADE_CNN[ic]; MAX_NORMAL_SESSAO = g["max_normal"]; rng = np.random.default_rng(SEMENTE)
Xc_tr, yc_tr = conjunto_cnn("treino"); Xc_va, yc_va = conjunto_cnn("validacao")
idx_rep = np.concatenate([rng.choice(np.where(yc_tr == c)[0], min(100, int((yc_tr == c).sum())), replace=False)
                          for c in range(4) if (yc_tr == c).any()])
conv = tf.lite.TFLiteConverter.from_keras_model(cnn)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = lambda: ([Xc_tr[i:i + 1]] for i in idx_rep)
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
conv._experimental_disable_per_channel_quantization_for_dense_layers = True
tfl = conv.convert(); open(f"{SAIDA}/cnn_insitu_int8.tflite", "wb").write(tfl)
it = Interpreter(model_content=tfl); it.allocate_tensors()
e_, s_ = it.get_input_details()[0], it.get_output_details()[0]
(se, ze), (ss, zs) = e_["quantization"], s_["quantization"]
def prob_int8(X):
    out = np.zeros((len(X), 4), np.float32)
    for i in range(len(X)):
        it.set_tensor(e_["index"], np.clip(np.round(X[i:i + 1] / se + ze), -128, 127).astype(np.int8)); it.invoke()
        out[i] = (it.get_tensor(s_["index"])[0].astype(np.float32) - zs) * ss
    return out
p8 = prob_int8(Xc_va); p32 = cnn.predict(Xc_va, verbose=0)
print(f"int8: {len(tfl) / 1024:.1f} KB | concordância 4 níveis: {100 * (p8.argmax(1) == p32.argmax(1)).mean():.2f}%")
print("F1 macro CNN na validação — float32:", round(f1_score(yc_va, p32.argmax(1), average="macro"), 4), "| int8:", round(f1_score(yc_va, p8.argmax(1), average="macro"), 4))
HP_ESC = {**HP, "lr_cnn": g["lr"], "peso_cnn": g["peso"], "max_normal": g["max_normal"], "peso_fusao": pf, "delta_normal": dl, "criterio": CRIT}
json.dump({"semente_pretreino": int(semente_cnn), "quantizacao": {"entrada": [float(se), float(ze)], "saida": [float(ss), float(zs)]},
           "normalizacao": "linha de base do nó (base.csv)", "linha_de_base": {"media": MU.tolist(), "desvio": SD.tolist()},
           "hp": HP_ESC, "versao": "v2 (busca 11b)", "sessoes": {s: d["papel"] for s, d in D.items()}},
          open(f"{SAIDA}/cnn_insitu_quantizacao.json", "w"), indent=1)
print("Modelos v2 gravados em", SAIDA)

Saved artifact at '/tmp/tmp0ef21qni'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 98, 40, 1), dtype=tf.float32, name='logmel')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  134355177407120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134355177409808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134355177407312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134355177414032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134355177414224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134355177407504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134355177413648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134355177409232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134355177412688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134355177413264: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


int8: 12.6 KB | concordância 4 níveis: 95.89%
F1 macro CNN na validação — float32: 0.7865 | int8: 0.7834
Modelos v2 gravados em /content/drive/MyDrive/dissertacao_tinyml/modelos/sim


## Próximos passos

1. **Notebook 12** sem mudança: avalia os modelos v2 no teste. Guarde também as saídas da v1 (já obtidas) para relatar as duas.
2. **Notebook 05b** e firmware v5: gere os cabeçalhos de novo (a CNN e o `FUSAO_B` mudaram).
3. **Texto (apêndice ou 8.5)**: a grade, o critério de seleção na validação e a tabela `busca_hiperparametros.csv` (as 10 melhores e a v1).
4. Para voltar à v1: copie o conteúdo de `modelos/sim_v1/` para `modelos/sim/`.